# Xây dựng và huấn luyện mô hình Random Forest Regressor và Gradient Boosting Regressor
Tập trung vào hai mô hình hồi quy **Random Forest Regressor** và **Gradient Boosting Regressor**, cùng phần **tinh chỉnh siêu tham số (hyperparameter tuning)**.

Mục tiêu là dự đoán biến mục tiêu dạng số (giá nhà) và so sánh cách hai mô hình ensemble hoạt động sau khi lựa chọn tham số phù hợp.

## 2. Random Forest Regressor

**Random Forest Regressor** là mô hình ensemble xây dựng nhiều cây quyết định và kết hợp dự đoán của các cây để tạo ra kết quả hồi quy ổn định hơn một cây đơn lẻ.

Một số siêu tham số thường cần quan tâm gồm `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf` và `max_features`. Việc điều chỉnh các tham số này giúp cân bằng giữa khả năng học dữ liệu và nguy cơ overfitting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline

## 1. Chuẩn bị dữ liệu

Phần này giữ lại các bước tiền xử lý và chia dữ liệu cần thiết cho quá trình huấn luyện. Thông thường ta có:

- `X_train`, `X_test`: các biến đầu vào dùng để huấn luyện và kiểm tra.
- `y_train`, `y_test`: giá trị mục tiêu tương ứng.
- Các bước preprocessing cần thiết để dữ liệu có thể đưa vào mô hình.

Việc giữ phần chuẩn bị dữ liệu giúp các đoạn code mô hình bên dưới có đầy đủ ngữ cảnh.

In [ ]:
n_train = train_feat.shape[0]
combined = pd.concat([train_feat, test_feat], axis=0, ignore_index=True)

cat_cols = combined.select_dtypes(exclude=[np.number]).columns.tolist()
print('Số cột phân loại cần encode:', len(cat_cols))

combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=True)

X_train_full = combined_encoded.iloc[:n_train, :].reset_index(drop=True)
X_test_final = combined_encoded.iloc[n_train:, :].reset_index(drop=True)

print('X_train_full:', X_train_full.shape)
print('X_test_final:', X_test_final.shape)

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y, test_size=0.2, random_state=42
)
print('X_tr:', X_tr.shape, ' X_val:', X_val.shape)

## 6. Xây dựng mô hình
### 6.1. Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_pred_val = rf.predict(X_val)

rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred_val))
rf_mae = mean_absolute_error(y_val, rf_pred_val)
rf_r2 = r2_score(y_val, rf_pred_val)

print(f'Random Forest -> RMSE: {rf_rmse:.2f} | MAE: {rf_mae:.2f} | R2: {rf_r2:.4f}')

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_tr.columns)\
    .sort_values(ascending=False).head(15)
plt.figure(figsize=(9,7))
sns.barplot(x=importances.values, y=importances.index, color='steelblue')
plt.title('Random Forest - Top 15 đặc trưng quan trọng nhất')
plt.xlabel('Mức độ quan trọng')
plt.show()

## 3. Gradient Boosting Regressor

**Gradient Boosting Regressor** xây dựng các cây tuần tự. Mỗi cây mới tập trung vào việc cải thiện sai số còn lại của mô hình trước đó.

Các siêu tham số quan trọng thường gồm `n_estimators`, `learning_rate`, `max_depth`, `min_samples_split`, `min_samples_leaf` và `subsample`.

Đặc biệt, `learning_rate` và `n_estimators` thường có mối quan hệ đánh đổi: learning rate nhỏ có thể cần nhiều cây hơn để đạt chất lượng tương đương.

### 6.2. Gradient Boosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42
)
gbr.fit(X_tr, y_tr)
gbr_pred_val = gbr.predict(X_val)

gbr_rmse = np.sqrt(mean_squared_error(y_val, gbr_pred_val))
gbr_mae = mean_absolute_error(y_val, gbr_pred_val)
gbr_r2 = r2_score(y_val, gbr_pred_val)

print(f'Gradient Boosting -> RMSE: {gbr_rmse:.2f} | MAE: {gbr_mae:.2f} | R2: {gbr_r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5.5))
axes[0].scatter(y_val, rf_pred_val, alpha=0.5, color='darkorange')
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--')
axes[0].set_title(f'Random Forest (R2={rf_r2:.3f})')
axes[0].set_xlabel('Giá thực tế'); axes[0].set_ylabel('Giá dự đoán')

axes[1].scatter(y_val, gbr_pred_val, alpha=0.5, color='seagreen')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--')
axes[1].set_title(f'Gradient Boosting (R2={gbr_r2:.3f})')
axes[1].set_xlabel('Giá thực tế'); axes[1].set_ylabel('Giá dự đoán')
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting'],
    'RMSE': [rf_rmse, gbr_rmse],
    'MAE': [rf_mae, gbr_mae],
    'R2': [rf_r2, gbr_r2]
})
comparison

In [ ]:
best_model_name = 'Gradient Boosting' if gbr_rmse < rf_rmse else 'Random Forest'
best_model = gbr if gbr_rmse < rf_rmse else rf
print('Mô hình được chọn:', best_model_name)

best_model.fit(X_train_full, y)
test_pred = best_model.predict(X_test_final)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': test_pred})
submission.to_csv('submission.csv', index=False)
submission.head()

## 9. Kết luận

- Sau khi xử lý dữ liệu thiếu và mã hóa one-hot, tập dữ liệu có 235 đặc trưng.
- **Gradient Boosting Regressor** cho kết quả tốt hơn **Random Forest Regressor** trên tập validation (RMSE thấp hơn, R² cao hơn), nên được chọn để dự đoán trên tập test.
- Các đặc trưng quan trọng nhất ảnh hưởng đến giá nhà gồm: OverallQual, GrLivArea, TotalBsmtSF, GarageCars, YearBuilt...
- File `submission.csv` chứa kết quả dự đoán cuối cùng để nộp lên Kaggle.

## 5. Huấn luyện mô hình với tham số được chọn

Sau khi quá trình tuning hoàn tất, mô hình tốt nhất được lấy từ đối tượng tìm kiếm tham số (ví dụ `best_estimator_`) và sử dụng để dự đoán trên tập kiểm tra.

Điểm quan trọng là **không chỉ nhìn vào kết quả trên tập huấn luyện**. Tập test được dùng để đánh giá khả năng tổng quát hóa của mô hình trên dữ liệu chưa được thấy.